# ARC NeuroGolf static ONNX solver

Reference layout adapted from the uploaded fill/additive-marking notebook. The task-specific modelling cell uses a semantic feature-tree or a symbolic reflection builder, not raw output-template lookup.

In [1]:
!rm -rf /kaggle/working/*
%reset -f

In [2]:
COMPETITION = '/kaggle/input/competitions/neurogolf-2026'

In [3]:
import importlib.util, subprocess, sys
missing=[p for p in ['onnx','onnxruntime','onnxscript','torch','numpy'] if importlib.util.find_spec(p) is None]
if missing:
    subprocess.check_call([sys.executable,'-m','pip','install','-q',*missing])
print('dependencies ok')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.7/18.7 MB 51.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 722.0/722.0 kB 14.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 166.8/166.8 kB 3.8 MB/s eta 0:00:00
dependencies ok


In [4]:
import json, os, time, hashlib, zipfile,  csv, base64
import glob, sys,math, random, collections,io,shutil
from pathlib import Path
import numpy as np
import torch
import onnx
import onnxruntime as ort
import torch, torch.nn as nn, torch.nn.functional as F
from collections import defaultdict
from onnx import shape_inference

In [5]:
TASK_ID = "task288"
SOURCE_JSON_NAME = "task288.json"
RULE_DESCRIPTION = "upward diagonal roof/ray from bottom composite bar"
CH = 10
H = W = 30
candidate_paths = [
    Path.cwd() / SOURCE_JSON_NAME,
    Path.cwd() / f"{TASK_ID}.json",
    Path("/mnt/data") / SOURCE_JSON_NAME,
    Path("/mnt/data") / f"{TASK_ID}.json",
    Path(COMPETITION) / SOURCE_JSON_NAME,
    Path(COMPETITION) / f"{TASK_ID}.json",
]
TASK_JSON = next((p for p in candidate_paths if p.exists()), None)
assert TASK_JSON is not None, "Could not locate task JSON. Checked: " + ", ".join(str(p) for p in candidate_paths)
OUT_DIR = Path.cwd() / f"{TASK_ID}_static_onnx"
OUT_DIR.mkdir(parents=True, exist_ok=True)
ONNX_PATH = OUT_DIR / f"{TASK_ID}.onnx"
SUBMISSION_PATH = Path.cwd() / "submission.zip"
SUMMARY_PATH = OUT_DIR / f"{TASK_ID}_validation_summary.json"

with TASK_JSON.open("r") as f:
    task = json.load(f)

print("task json:", TASK_JSON)
len(task["train"]), len(task["test"]), len(task.get("arc-gen", []))

task json: /kaggle/input/competitions/neurogolf-2026/task288.json


(4, 1, 262)

In [6]:

def grid_to_tensor_zero_padded(grid, h=H, w=W, ch=CH):
    """Convert ARC grid to [1,10,30,30].
    Inside real grid: one-hot, including background color 0.
    Outside real grid: all-zero across all channels, not color-0 background.
    """
    x = np.zeros((1, ch, h, w), dtype=np.float32)
    for r, row in enumerate(grid):
        for c, v in enumerate(row):
            x[0, int(v), r, c] = 1.0
    return x

def show_grid(g):
    return "\n".join("".join(str(v) if v else "." for v in row) for row in g)


def _bbox_points(mask):
    rr, cc = np.where(mask)
    return int(rr.min()), int(rr.max()), int(cc.min()), int(cc.max())

def python_rule_task288(grid):
    arr = np.array(grid, dtype=np.int64)
    out = arr.copy()
    h, w = arr.shape
    top_row = h - 2
    center = (w - 1) // 2
    draw = int(arr[h - 1, center])
    nz = np.where(arr[top_row] != 0)[0]
    L, R = int(nz.min()), int(nz.max())
    for r in range(top_row):
        dl = top_row - r
        c1, c2 = L - dl, R + dl
        if 0 <= c1 < w: out[r, c1] = draw
        if 0 <= c2 < w: out[r, c2] = draw
    return out.tolist()

def python_rule_task248(grid):
    arr = np.array(grid, dtype=np.int64)
    h, w = arr.shape
    color = int(arr[arr != 0][0])
    out = np.zeros_like(arr)
    period = 2 * (w - 1)
    for r in range(h):
        d = h - 1 - r
        p = d % period
        c = min(p, period - p)
        out[r, c] = color
    return out.tolist()

def python_rule_task268(grid):
    arr = np.array(grid, dtype=np.int64)
    out = arr.copy()
    h, w = arr.shape
    obj = arr != 0
    r0, r1, c0, c1 = _bbox_points(obj)
    inside = np.zeros_like(obj)
    inside[r0:r1+1, c0:c1+1] = True
    fill = inside & (~obj)
    top_gap = [c for c in range(c0, c1+1) if arr[r0, c] == 0]
    bot_gap = [c for c in range(c0, c1+1) if arr[r1, c] == 0]
    left_gap = [r for r in range(r0, r1+1) if arr[r, c0] == 0]
    right_gap = [r for r in range(r0, r1+1) if arr[r, c1] == 0]
    if top_gap:
        g0, g1 = min(top_gap), max(top_gap)
        for r in range(0, r0):
            d = r0 - r
            for c in range(w):
                if g0 <= c <= g1 or c == g0 - d or c == g1 + d:
                    fill[r, c] = True
    if bot_gap:
        g0, g1 = min(bot_gap), max(bot_gap)
        for r in range(r1 + 1, h):
            d = r - r1
            for c in range(w):
                if g0 <= c <= g1 or c == g0 - d or c == g1 + d:
                    fill[r, c] = True
    if left_gap:
        g0, g1 = min(left_gap), max(left_gap)
        for c in range(0, c0):
            d = c0 - c
            for r in range(h):
                if g0 <= r <= g1 or r == g0 - d or r == g1 + d:
                    fill[r, c] = True
    if right_gap:
        g0, g1 = min(right_gap), max(right_gap)
        for c in range(c1 + 1, w):
            d = c - c1
            for r in range(h):
                if g0 <= r <= g1 or r == g0 - d or r == g1 + d:
                    fill[r, c] = True
    out[fill] = 4
    return out.tolist()

def python_rule_task280(grid):
    arr = np.array(grid, dtype=np.int64)
    h, w = arr.shape
    out = arr.copy()
    fg = (arr == 2) | (arr == 3)
    left = np.zeros_like(arr, dtype=np.int64)
    right = np.zeros_like(arr, dtype=np.int64)
    up = np.zeros_like(arr, dtype=np.int64)
    down = np.zeros_like(arr, dtype=np.int64)
    for r in range(h):
        for c in range(w):
            left[r, c] = (left[r, c-1] + 1 if c > 0 else 1) if fg[r, c] else 0
        for c in range(w-1, -1, -1):
            right[r, c] = (right[r, c+1] + 1 if c < w-1 else 1) if fg[r, c] else 0
    for c in range(w):
        for r in range(h):
            up[r, c] = (up[r-1, c] + 1 if r > 0 else 1) if fg[r, c] else 0
        for r in range(h-1, -1, -1):
            down[r, c] = (down[r+1, c] + 1 if r < h-1 else 1) if fg[r, c] else 0
    for r, c in zip(*np.where(arr == 2)):
        rowrun = left[r, c] + right[r, c] - 1
        colrun = up[r, c] + down[r, c] - 1
        if rowrun > colrun:
            span = colrun - 1
            c0, c1 = c - span, c + span
            if up[r, c] == 1:
                top = r - up[r, c] + 1
                rows = range(0, top)
            else:
                bottom = r + down[r, c] - 1
                rows = range(bottom + 1, h)
            for rr in rows:
                for cc in range(max(0, c0), min(w, c1 + 1)):
                    if out[rr, cc] == 0:
                        out[rr, cc] = 2 if cc == c else 3
        else:
            span = rowrun - 1
            r0, r1 = r - span, r + span
            if left[r, c] == 1:
                left_bound = c - left[r, c] + 1
                cols = range(0, left_bound)
            else:
                right_bound = c + right[r, c] - 1
                cols = range(right_bound + 1, w)
            for rr in range(max(0, r0), min(h, r1 + 1)):
                for cc in cols:
                    if out[rr, cc] == 0:
                        out[rr, cc] = 2 if rr == r else 3
    return out.tolist()

def python_rule_task284(grid):
    arr = np.array(grid, dtype=np.int64)
    h, w = arr.shape
    out = np.zeros_like(arr)
    obj = arr != 0
    r0, r1, c0, c1 = _bbox_points(obj)
    if c0 == c1:  # vertical anchors
        c = c0
        top_color = int(arr[r0, c])
        bot_color = int(arr[r1, c])
        t = (r0 + r1 - 3) // 2
        for r in range(r0, t + 1): out[r, c] = top_color
        for c2 in range(c - 2, c + 3):
            if 0 <= c2 < w: out[t, c2] = top_color
        for c2 in [c - 2, c + 2]:
            if 0 <= t + 1 < h and 0 <= c2 < w: out[t + 1, c2] = top_color
            if 0 <= t + 2 < h and 0 <= c2 < w: out[t + 2, c2] = bot_color
        for c2 in range(c - 2, c + 3):
            if 0 <= t + 3 < h and 0 <= c2 < w: out[t + 3, c2] = bot_color
        for r in range(t + 3, r1 + 1): out[r, c] = bot_color
    else:  # horizontal anchors
        r = r0
        left_color = int(arr[r, c0])
        right_color = int(arr[r, c1])
        m = (c0 + c1 - 3) // 2
        mr = m + 3
        for c in range(c0, m + 1): out[r, c] = left_color
        for rr in range(r - 1, r + 2):
            if 0 <= rr < h: out[rr, m] = left_color
        for rr in [r - 2, r + 2]:
            for c in range(m, m + 2):
                if 0 <= rr < h and 0 <= c < w: out[rr, c] = left_color
        for c in range(mr, c1 + 1): out[r, c] = right_color
        for rr in range(r - 1, r + 2):
            if 0 <= rr < h: out[rr, mr] = right_color
        for rr in [r - 2, r + 2]:
            for c in range(mr - 1, mr + 1):
                if 0 <= rr < h and 0 <= c < w: out[rr, c] = right_color
    return out.tolist()

def python_rule(grid):
    return globals()[f"python_rule_{TASK_ID}"](grid)


for split in ["train", "test", "arc-gen"]:
    ok = sum(python_rule(ex["input"]) == ex["output"] for ex in task.get(split, []))
    print(split, ok, "/", len(task.get(split, [])))


train 4 / 4
test 1 / 1
arc-gen 262 / 262


In [7]:
class Base(nn.Module):
    def __init__(self,h=H,w=W):
        super().__init__()
        rr=torch.arange(h,dtype=torch.float32).view(1,1,h,1).expand(1,1,h,w)
        cc=torch.arange(w,dtype=torch.float32).view(1,1,1,w).expand(1,1,h,w)
        self.register_buffer('R',rr); self.register_buffer('C',cc)
    def color_map(self,x):
        return torch.argmax(x, dim=1, keepdim=True).float()
    def active(self,x):
        return (x.sum(dim=1, keepdim=True)>0.5).float()
    def to_onehot(self,colors,active):
        outs=[]
        for k in range(10):
            outs.append(((colors - float(k)).abs()<0.25).float()*active)
        return torch.cat(outs, dim=1)
    def dims(self,active):
        row_active=(active.sum(dim=3,keepdim=True)>0.5).float()
        col_active=(active.sum(dim=2,keepdim=True)>0.5).float()
        hn=row_active.sum(dim=(2,3),keepdim=True)
        wn=col_active.sum(dim=(2,3),keepdim=True)
        return hn,wn
    def bbox(self,obj):
        big=torch.tensor(99.0, device=obj.device)
        neg=torch.tensor(-99.0, device=obj.device)
        minR=torch.amin(torch.where(obj>0.5,self.R,big), dim=(2,3), keepdim=True)
        maxR=torch.amax(torch.where(obj>0.5,self.R,neg), dim=(2,3), keepdim=True)
        minC=torch.amin(torch.where(obj>0.5,self.C,big), dim=(2,3), keepdim=True)
        maxC=torch.amax(torch.where(obj>0.5,self.C,neg), dim=(2,3), keepdim=True)
        return minR,maxR,minC,maxC


class Task288Model(Base):
    def forward(self,x):
        active=self.active(x); colors=self.color_map(x)
        nonzero=(colors>0.5).float()*active
        hn,wn=self.dims(active)
        top_row=hn-2.0
        center=torch.floor((wn-1.0)/2.0)
        bottom_center=((self.R-(hn-1.0)).abs()<0.25).float()*((self.C-center).abs()<0.25).float()*active
        draw_color=(colors*bottom_center).sum(dim=(2,3),keepdim=True)
        topseg=nonzero*((self.R-top_row).abs()<0.25).float()
        big=torch.tensor(99.0,device=x.device); neg=torch.tensor(-99.0,device=x.device)
        L=torch.amin(torch.where(topseg>0.5,self.C,big),dim=(2,3),keepdim=True)
        Rr=torch.amax(torch.where(topseg>0.5,self.C,neg),dim=(2,3),keepdim=True)
        fill=(self.R<top_row).float()*(
            (((self.C-self.R)-(L-top_row)).abs()<0.25).float()+
            (((self.C+self.R)-(Rr+top_row)).abs()<0.25).float()
        )
        fill=(fill>0.5).float()*active
        out=colors*(1.0-fill)+draw_color*fill
        return self.to_onehot(out,active)


model = Task288Model().eval()


In [8]:

dummy = torch.from_numpy(grid_to_tensor_zero_padded(task["test"][0]["input"]))

torch.onnx.export(
    model,
    dummy,
    str(ONNX_PATH),
    input_names=["input"],
    output_names=["output"],
    opset_version=17,
    do_constant_folding=True,
    dynamic_axes=None,
    dynamo=False,
)

onnx_model = onnx.load(str(ONNX_PATH))
onnx_model = onnx.shape_inference.infer_shapes(onnx_model)
onnx.save(onnx_model, str(ONNX_PATH))
onnx.checker.check_model(str(ONNX_PATH))

ONNX_PATH, ONNX_PATH.stat().st_size


/tmp/ipykernel_16/2818671719.py:3: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter has become the default. Learn more about the new export logic: https://docs.pytorch.org/docs/stable/onnx_export.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html
  torch.onnx.export(
/tmp/ipykernel_16/1553485097.py:42: TracerWarning: torch.tensor results are registered as constants in the trace. You can safely ignore this warning if you use this function to create tensors out of constant variables that would be the same every time you call this function. In any other case, this might cause the trace to be incorrect.
  big=torch.tensor(99.0,device=x.device); neg=torch.tensor(-99.0,device=x.device)


(PosixPath('/kaggle/working/task288_static_onnx/task288.onnx'), 32162)

In [9]:

def vi_shape(vi):
    dims = []
    for d in vi.type.tensor_type.shape.dim:
        if d.dim_value:
            dims.append(int(d.dim_value))
        elif d.dim_param:
            dims.append(str(d.dim_param))
        else:
            dims.append(None)
    return dims

onnx_model = onnx.load(str(ONNX_PATH))
ops = collections.Counter(node.op_type for node in onnx_model.graph.node)
forbidden = {"Loop", "Scan", "NonZero", "Unique", "Script", "Function"}
empty_inputs = [
    (node.name, node.op_type, list(node.input))
    for node in onnx_model.graph.node
    if any(inp == "" for inp in node.input)
]
bad_shapes = []
for vi in list(onnx_model.graph.input) + list(onnx_model.graph.value_info) + list(onnx_model.graph.output):
    shp = vi_shape(vi)
    if any(d is None or isinstance(d, str) for d in shp):
        bad_shapes.append((vi.name, shp))

print("input shape:", vi_shape(onnx_model.graph.input[0]))
print("output shape:", vi_shape(onnx_model.graph.output[0]))
print("ONNX size:", ONNX_PATH.stat().st_size)
print("ops:", dict(ops))
print("forbidden ops:", sorted(forbidden & set(ops)))
print("empty optional inputs:", len(empty_inputs))
print("non-static tensor shapes:", len(bad_shapes))

assert vi_shape(onnx_model.graph.input[0]) == [1, 10, 30, 30]
assert vi_shape(onnx_model.graph.output[0]) == [1, 10, 30, 30]
assert not (forbidden & set(ops))
assert not empty_inputs
assert not bad_shapes
assert ONNX_PATH.stat().st_size < 1_440_000


input shape: [1, 10, 30, 30]
output shape: [1, 10, 30, 30]
ONNX size: 32162
ops: {'Constant': 42, 'ReduceSum': 6, 'Greater': 6, 'Cast': 22, 'ArgMax': 1, 'Mul': 19, 'Sub': 20, 'Div': 1, 'Floor': 1, 'Abs': 15, 'Less': 16, 'Where': 2, 'ReduceMin': 1, 'ReduceMax': 1, 'Add': 3, 'Concat': 1}
forbidden ops: []
empty optional inputs: 0
non-static tensor shapes: 0


In [10]:

sess = ort.InferenceSession(str(ONNX_PATH), providers=["CPUExecutionProvider"])

def validate_examples(examples):
    tensor_ok = 0
    grid_ok = 0
    outside_zero_ok = 0
    bad = []
    for i, ex in enumerate(examples):
        x = grid_to_tensor_zero_padded(ex["input"])
        y = sess.run(None, {"input": x})[0]
        exp = grid_to_tensor_zero_padded(ex["output"])
        pred_bin = (y > 0.5).astype(np.float32)

        if np.array_equal(pred_bin, exp):
            tensor_ok += 1
        else:
            bad.append(i)

        h, w = len(ex["output"]), len(ex["output"][0])
        pred_grid = pred_bin[0, :, :h, :w].argmax(axis=0).astype(np.int64).tolist()
        if pred_grid == ex["output"]:
            grid_ok += 1

        active = x.sum(axis=1, keepdims=True) > 0.5
        if np.all(np.abs(y * (~active)) < 1e-5):
            outside_zero_ok += 1

    return {
        "tensor_exact_zero_padded": [tensor_ok, len(examples)],
        "grid_argmax_inside_canvas": [grid_ok, len(examples)],
        "outside_active_all_channels_zero": [outside_zero_ok, len(examples)],
        "bad_indices": bad[:10],
    }

def deterministic_holdout(examples, fraction=0.60):
    idx = list(range(len(examples)))
    rng = random.Random(20260707)
    rng.shuffle(idx)
    n = int(math.ceil(len(idx) * fraction))
    return [examples[i] for i in idx[:n]], idx[:n]

arc_holdout, arc_holdout_indices = deterministic_holdout(task.get("arc-gen", []), 0.60)
summary = {
    "task_id": TASK_ID,
    "rule": RULE_DESCRIPTION,
    "onnx_path": str(ONNX_PATH),
    "onnx_size_bytes": ONNX_PATH.stat().st_size,
    "input_shape": vi_shape(onnx_model.graph.input[0]),
    "output_shape": vi_shape(onnx_model.graph.output[0]),
    "ops": dict(ops),
    "forbidden_ops": sorted(forbidden & set(ops)),
    "empty_optional_inputs": len(empty_inputs),
    "non_static_tensor_shapes": len(bad_shapes),
    "arc_gen_holdout_fraction": 0.60,
    "arc_gen_holdout_count": len(arc_holdout),
    "arc_gen_holdout_indices_first_20": arc_holdout_indices[:20],
    "validation": {
        "train": validate_examples(task["train"]),
        "test": validate_examples(task["test"]),
        "arc-gen-60pct-holdout": validate_examples(arc_holdout),
        "arc-gen-full": validate_examples(task.get("arc-gen", [])),
    },
}

print(json.dumps(summary, indent=2)[:5000])
with SUMMARY_PATH.open("w") as f:
    json.dump(summary, f, indent=2)


{
  "task_id": "task288",
  "rule": "upward diagonal roof/ray from bottom composite bar",
  "onnx_path": "/kaggle/working/task288_static_onnx/task288.onnx",
  "onnx_size_bytes": 32162,
  "input_shape": [
    1,
    10,
    30,
    30
  ],
  "output_shape": [
    1,
    10,
    30,
    30
  ],
  "ops": {
    "Constant": 42,
    "ReduceSum": 6,
    "Greater": 6,
    "Cast": 22,
    "ArgMax": 1,
    "Mul": 19,
    "Sub": 20,
    "Div": 1,
    "Floor": 1,
    "Abs": 15,
    "Less": 16,
    "Where": 2,
    "ReduceMin": 1,
    "ReduceMax": 1,
    "Add": 3,
    "Concat": 1
  },
  "forbidden_ops": [],
  "empty_optional_inputs": 0,
  "non_static_tensor_shapes": 0,
  "arc_gen_holdout_fraction": 0.6,
  "arc_gen_holdout_count": 158,
  "arc_gen_holdout_indices_first_20": [
    115,
    162,
    20,
    138,
    9,
    13,
    136,
    163,
    248,
    146,
    144,
    55,
    127,
    126,
    28,
    35,
    227,
    216,
    239,
    241
  ],
  "validation": {
    "train": {
      "tensor_exact

In [11]:

with zipfile.ZipFile(SUBMISSION_PATH, "w", compression=zipfile.ZIP_DEFLATED) as z:
    z.write(ONNX_PATH, arcname=f"{TASK_ID}.onnx")

print("Wrote:", SUBMISSION_PATH)
print("Zip contents:", zipfile.ZipFile(SUBMISSION_PATH).namelist())
assert zipfile.ZipFile(SUBMISSION_PATH).namelist() == [f"{TASK_ID}.onnx"]


Wrote: /kaggle/working/submission.zip
Zip contents: ['task288.onnx']
